In [ ]:
import pandas as pd
import kagglehub
import shutil
import os
from pathlib import Path
import time
import glob

from nba_api.stats.endpoints import PlayerGameLogs, CommonPlayerInfo

## 1. Directory Setup

In [ ]:
Path("../data/raw").mkdir(parents=True, exist_ok=True)
Path("../data/processed").mkdir(parents=True, exist_ok=True)
Path("../data/raw/game_logs").mkdir(parents=True, exist_ok=True)

## 2. Getting NBA Injury Data

In [2]:
# Download to kagglehub cache first
path = kagglehub.dataset_download("loganlauton/nba-injury-stats-1951-2023")

# Copy files to data/raw folder
dest = "../data/raw"
os.makedirs(dest, exist_ok=True)

for file in os.listdir(path):
    dest_file = os.path.join(dest, file)
    if not os.path.exists(dest_file):
        shutil.copy(os.path.join(path, file), dest)
        print(f"Copied: {file}")
    else:
        print(f"Skipped (already exists): {file}")

print("Files copied to:", dest)

Skipped (already exists): NBA Player Injury Stats(1951 - 2023).csv
Files copied to: ../data/raw


In [ ]:
df_injury = pd.read_csv(f'{dest}/{file}')

# drop the junk index
df_injury = df_injury.drop(columns=["Unnamed: 0"])

# filter to your window
df_injury["Date"] = pd.to_datetime(df_injury["Date"])
df_injury = df_injury[df_injury["Date"].dt.year >= 2015]
df_injury = df_injury.reset_index(drop=True)

print(f"Shape: {df_injury.shape}")
print(f"Date range: {df_injury['Date'].min()} → {df_injury['Date'].max()}")
print(f"\nNotes value counts (top 20):\n{df_injury['Notes'].value_counts().head(20)}")
print(f"\nNull counts:\n{df_injury.isnull().sum()}")

Shape: (16316, 5)
Date range: 2015-01-01 00:00:00 → 2023-04-16 00:00:00

Notes value counts (top 20):
Notes
activated from IL                                    7724
placed on IL                                         2175
placed on IL with NBA health and safety protocols     522
placed on IL with illness                             443
placed on IL with sprained left ankle                 284
placed on IL with sprained right ankle                280
placed on IL with sore left knee                      180
placed on IL with sore right knee                     177
placed on IL for rest                                 154
placed on IL with concussion                          113
placed on IL with left knee injury                    105
placed on IL with sore lower back                      91
placed on IL with sore left ankle                      84
placed on IL with right knee injury                    79
placed on IL with sore right ankle                     71
placed on IL with left

In [5]:
df_injury.head(10)

,Date,Team,Acquired,Relinquished,Notes
0,2015-01-01,Bulls,Kirk Hinrich,NaN,activated from IL
1,2015-01-01,Kings,NaN,Omri Casspi,placed on IL with left knee injury
2,2015-01-02,Bucks,NaN,Larry Sanders (b. 1988-11-21),placed on IL
3,2015-01-02,Knicks,NaN,Tim Hardaway Jr.,placed on IL with concussion
4,2015-01-02,Knicks,Cleanthony Early,NaN,activated from IL
5,2015-01-02,Magic,NaN,Maurice Harkless / Moe Harkless,placed on IL
6,2015-01-02,Magic,Roy Devyn Marble / Roy Marble (Devyn),NaN,activated from IL
7,2015-01-02,Nets,NaN,Cory Jefferson,placed on IL
8,2015-01-02,Nets,Kevin Garnett,NaN,activated from IL
9,2015-01-02,Pistons,NaN,Spencer Dinwiddie,placed on IL


## 3. Fetching Player Game Logs
### From 2015-16 season to 2022-23 season. 
### A total of eight seasons.

In [ ]:
# Directory Setup
RAW_DIR = Path("../data/raw/game_logs")

seasons = [
    "2015-16", "2016-17", "2017-18", "2018-19",
    "2019-20", "2020-21", "2021-22", "2022-23"
]

season_types = ["Regular Season", "Playoffs"]

In [ ]:
def fetch_season_logs(season, season_type, retries=3):
    for attempt in range(retries):
        try:
            logs = PlayerGameLogs(
                season_nullable=season,
                season_type_nullable=season_type,
                timeout=120,
            )
            return logs.get_data_frames()[0]
        except Exception as e:
            print(f"  ⚠️ Attempt {attempt + 1} failed: {e}")
            if attempt < retries - 1:
                wait = (attempt + 1) * 5
                print(f"  Retrying in {wait}s...")
                time.sleep(wait)
    return None


for season in seasons:
    for season_type in season_types:
        label = season_type.lower().replace(" ", "_")
        out_path = RAW_DIR / f"player_game_logs_{season.replace('-', '_')}_{label}.csv"

        if out_path.exists():
            print(f"Skipping {season} {season_type} — already exists")
            continue

        print(f"Fetching {season} {season_type}...")
        df = fetch_season_logs(season, season_type)

        if df is None:
            print(f"All retries FAILED for {season} {season_type}")
        elif df.empty:
            print(f"No data for {season} {season_type}")
        else:
            df.to_csv(out_path, index=False)
            print(f"✅ {len(df)} rows → {out_path.name}")

        time.sleep(3)

print("Done!")

Skipping 2015-16 Regular Season — already exists
Skipping 2015-16 Playoffs — already exists
Skipping 2016-17 Regular Season — already exists
Skipping 2016-17 Playoffs — already exists
Skipping 2017-18 Regular Season — already exists
Skipping 2017-18 Playoffs — already exists
Skipping 2018-19 Regular Season — already exists
Skipping 2018-19 Playoffs — already exists
Skipping 2019-20 Regular Season — already exists
Skipping 2019-20 Playoffs — already exists
Skipping 2020-21 Regular Season — already exists
Skipping 2020-21 Playoffs — already exists
Skipping 2021-22 Regular Season — already exists
Skipping 2021-22 Playoffs — already exists
Skipping 2022-23 Regular Season — already exists
Skipping 2022-23 Playoffs — already exists
Done!


## 4. Combine and Save Game Logs

In [ ]:
files = glob.glob("../data/raw/game_logs/**/*.csv", recursive=True)

dfs = []
for f in files:
    temp = pd.read_csv(f)
    temp["is_playoff"] = 1 if "playoff" in f else 0
    dfs.append(temp)

df_game_logs = pd.concat(dfs, ignore_index=True)
df_game_logs.to_csv("../data/processed/player_game_logs_all.csv", index=False)
print(df_game_logs.shape)

(216109, 71)


## 5. Fetching Player Birthdates

In [ ]:
player_ids = df_game_logs["PLAYER_ID"].unique()

print(f"Unique players to fetch: {len(player_ids)}")

Unique players to fetch: 1248


In [ ]:
records = []
failed = []

for i, pid in enumerate(player_ids):
    try:
        info = CommonPlayerInfo(player_id=pid, timeout=60)
        row = info.get_data_frames()[0]
        records.append({
            "PLAYER_ID":   pid,
            "PLAYER_NAME": row["DISPLAY_FIRST_LAST"].iloc[0],
            "BIRTHDATE":   row["BIRTHDATE"].iloc[0]
        })
        if i % 50 == 0:
            print(f"  {i}/{len(player_ids)} fetched...")
        time.sleep(0.9)
    except Exception as e:
        print(f"FAILED for PLAYER_ID {pid}: {e}")
        failed.append(pid)
        time.sleep(2)

print(f"\nSuccessfully fetched: {len(records)}")
print(f"Failed: {len(failed)} — {failed}")

df_birthdates = pd.DataFrame(records)
df_birthdates["BIRTHDATE"] = pd.to_datetime(df_birthdates["BIRTHDATE"])
df_birthdates.to_csv("../data/raw/player_birthdates.csv", index=False)
print("Saved: player_birthdates.csv")

  0/1248 fetched...
  50/1248 fetched...
  100/1248 fetched...
  150/1248 fetched...
  200/1248 fetched...
  250/1248 fetched...
  300/1248 fetched...
  350/1248 fetched...
  400/1248 fetched...
  450/1248 fetched...
  500/1248 fetched...
  550/1248 fetched...
  600/1248 fetched...
  650/1248 fetched...
  700/1248 fetched...
  750/1248 fetched...
  800/1248 fetched...
  850/1248 fetched...
  900/1248 fetched...
  950/1248 fetched...
  1000/1248 fetched...
  1050/1248 fetched...
  1100/1248 fetched...
  1150/1248 fetched...
  1200/1248 fetched...

Successfully fetched: 1248
Failed: 0 — []
Saved: player_birthdates.csv


## Data Validation

In [ ]:
print("=== Injury Data ===")
print(df_injury.shape)
print(df_injury.dtypes)

print("\n=== Game Logs ===")
print(df_game_logs.shape)
print(df_game_logs.isnull().sum())
print("Playoff rows:", df_game_logs[df_game_logs["is_playoff"] == 1].shape[0])
print("Regular season rows:", df_game_logs[df_game_logs["is_playoff"] == 0].shape[0])

print("\n=== Birthdates ===")
print(df_birthdates.shape)
df_birthdates.head()